In [ ]:
# imports

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [ ]:
# environment

load_dotenv(override=True)
DB = "products_vectorstore"

In [ ]:
# Log in to HuggingFace
# If you don't have a HuggingFace account, you can set one up for free at www.huggingface.co
# And then add the HF_TOKEN to your .env file as explained in the project README

hf_token = os.environ['HUGGING_FACE_API']
login(token=hf_token, add_to_git_credential=False)

In [ ]:
LITE_MODE = True

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:

vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

In [ ]:
client = chromadb.PersistentClient(path=DB)
collection_name = "products"
existing=[collection.name for collection in client.list_collections()]

if collection_name not in existing:
    collection=client.create_collection(collection_name)
    for i in tqdm(range(0,len(train),1000)):
        documents=[item.summary for item in train[i:i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids=ids[:len(documents)]
        collection.add(ids=ids,documents=documents,embeddings=vectors, metadatas=metadatas)
collection = client.get_or_create_collection(collection_name)

In [ ]:
# It is very fun turning this up to 800_000 and seeing the full dataset visualized,
# but it almost crashes my box every time so do that at your own risk!! 10_000 is safe!

MAXIMUM_DATAPOINTS = 10_000

In [ ]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [ ]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
test[0]

In [ ]:
def vector(item):
    return encoder.encode(item.summary)

In [ ]:
vector(test[0])

In [ ]:
def find_similars(item):
    vec=vector(item)
    results=collection.query(query_embeddings=vec.astype(float).tolist(),n_results=5)
    documents=results['documents'][0][:]
    prices=[m['price']for m in results['metadatas'][0][:]]
    return documents,prices

In [ ]:
find_similars(test[0])

In [ ]:
def make_context(similars,prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar,prices in zip(similars,prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${prices:.2f}\n\n"
        return message

In [ ]:
def make_message(item,similars,prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message+=make_context(similars,prices)
    return [{"role":"user","content":message}]

In [ ]:
documents, prices = find_similars(test[0])


In [ ]:
print(make_message(test[0], documents,prices)[0]['content'])

In [ ]:
from openai import OpenAI
load_dotenv(override=True)
client=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [ ]:
def router(item):
    documents,prices=find_similars(item)
    response=client.chat.completions.create(model='openrouter/free',messages=make_message(item,documents, prices),seed=42)
    return response.choices[0].message.content


In [ ]:
# How much does our favorite distortion pedal cost?

test[0].price

In [ ]:
router(test[0])

In [ ]:
evaluate(router, test)

In [ ]:
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [ ]:
def specialist(item):
    return pricer.price.remote(item.summary)

In [ ]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

In [ ]:
def ensemble(item):
    price1 = get_price(router(item))
    price2 = specialist(item)
    return price1 * 0.8 + price2 * 0.2


In [ ]:
ensemble(test[0])

In [ ]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)